In [ ]:
import requests
import json
import time
import csv
from datetime import datetime
from pathlib import Path
import re
from collections import Counter
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
class ChroniclingAmericaScraper:
    """
    A scraper for building datasets from the Chronicling America newspaper collection.
    Searches for articles and retrieves full text using the Library of Congress APIs.
    """
    
    def __init__(self, output_dir="chronicling_america_data"):
        self.base_search_url = "https://www.loc.gov/collections/chronicling-america/"
        self.text_service_url = "https://tile.loc.gov/text-services/word-coordinates-service"
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)
        
        # Rate limiting parameters (stay well under limits)
        self.min_request_interval = 3  # seconds between requests
        self.last_request_time = 0
        
    def _rate_limit(self):
        """Enforce rate limiting between API requests."""
        elapsed = time.time() - self.last_request_time
        if elapsed < self.min_request_interval:
            time.sleep(self.min_request_interval - elapsed)
        self.last_request_time = time.time()
    
    def search_articles(self, query=None, start_date=None, end_date=None, 
                       state=None, max_results=100, search_type="PHRASE", 
                       custom_url=None):
        """
        Search for newspaper articles in Chronicling America.
        
        Args:
            query: Search term(s) (e.g., "Chinese students")
            start_date: Start date in YYYY-MM-DD format
            end_date: End date in YYYY-MM-DD format
            state: State to filter by (e.g., "new york")
            max_results: Maximum number of results to retrieve
            search_type: Type of search operation:
                - "PHRASE" (default): This exact phrase
                - "AND": All of these words
                - "OR": Any of these words
                - "~5": These words within 5 words of each other
                - "~10": These words within 10 words of each other
            custom_url: Use a pre-defined search URL from the website
            
        Returns:
            List of article metadata dictionaries
        """
        if custom_url:
            print(f"Using custom search URL")
            # Parse the custom URL to extract base and params
            from urllib.parse import urlparse, parse_qs
            parsed = urlparse(custom_url)
            params = parse_qs(parsed.query)
            # Convert lists to single values
            params = {k: v[0] if len(v) == 1 else v for k, v in params.items()}
            # Ensure we have fo=json
            params['fo'] = 'json'
            params['c'] = 100  # Results per page
        else:
            print(f"Searching for: '{query}' (search type: {search_type})")
            # Build query parameters
            params = {
                'q': query,
                'dl': 'page',  # Get page-level results
                'fo': 'json',
                'c': 100,  # Results per page
                'ops': search_type  # Search operation type
            }
        
        if start_date:
            params['start_date'] = start_date
        if end_date:
            params['end_date'] = end_date
        if state:
            params['location_state'] = state.lower()
        
        all_results = []
        page = 1
        
        while len(all_results) < max_results:
            self._rate_limit()
            
            params['sp'] = page
            print(f"Fetching page {page}...")
            
            try:
                response = requests.get(self.base_search_url, params=params, timeout=30)
                response.raise_for_status()
                data = response.json()
                
                # Extract results
                results = data.get('results', [])
                if not results:
                    print("No more results found.")
                    break
                
                all_results.extend(results)
                print(f"Retrieved {len(results)} results (total: {len(all_results)})")
                
                # Check if there are more pages
                pagination = data.get('pagination', {})
                if not pagination.get('next'):
                    print("Reached last page.")
                    break
                    
                page += 1
                
            except Exception as e:
                print(f"Error fetching page {page}: {e}")
                break
        
        # Limit to max_results
        all_results = all_results[:max_results]
        print(f"\nTotal articles found: {len(all_results)}")
        return all_results
    
    def extract_segment_info(self, item):
        """
        Extract segment path and format from item metadata.
        
        Args:
            item: Article metadata dictionary
            
        Returns:
            Tuple of (segment_path, format) or (None, None)
        """
        # Method 1: Check if full_text is already in the search results
        if 'full_text' in item and item['full_text']:
            return 'EMBEDDED', 'embedded'
        
        # Method 2: Look in image_url array for text-services URL with segment
        image_urls = item.get('image_url', [])
        for url in image_urls:
            if 'text-services/word-coordinates-service' in url and 'segment=' in url:
                # Extract segment parameter from URL
                match = re.search(r'segment=([^&]+)', url)
                if match:
                    segment = match.group(1)
                    # URL decode if needed
                    segment = segment.replace('%2F', '/')
                    return segment, 'alto_xml'
        
        # Method 3: Check word_coordinates_url field
        word_coords_url = item.get('word_coordinates_url', '')
        if word_coords_url and 'segment=' in word_coords_url:
            match = re.search(r'segment=([^&]+)', word_coords_url)
            if match:
                segment = match.group(1)
                segment = segment.replace('%2F', '/')
                return segment, 'alto_xml'
        
        return None, None
    
    def get_full_text(self, segment_path, format_type='alto_xml', search_query=None, 
                     include_snippet=False):
        """
        Retrieve full text OCR for a newspaper page.
        
        Args:
            segment_path: Path to the document segment
            format_type: Format of the segment (e.g., 'alto_xml')
            search_query: Search terms to highlight (e.g., "chinese students")
            include_snippet: If True, get relevant snippet instead of full text
            
        Returns:
            Dictionary with full text and metadata, or None on error
        """
        self._rate_limit()
        
        # Handle both storage-services and regular service paths
        if segment_path.startswith('storage-services/'):
            segment_path = '/' + segment_path
        
        params = {
            'segment': segment_path,
            'format': format_type,
        }
        
        # Add full_text or relevant_snippet
        if include_snippet and search_query:
            params['relevant_snippet'] = 1
            params['q'] = search_query
        else:
            params['full_text'] = 1
        
        try:
            response = requests.get(self.text_service_url, params=params, timeout=30)
            response.raise_for_status()
            data = response.json()
            
            # Extract the full text from the response
            # The key might be the segment path with or without leading slash
            for key in [segment_path, segment_path.lstrip('/')]:
                if key in data:
                    return data[key]
            
            # If exact match not found, try to find any key that contains the segment
            for key in data.keys():
                if segment_path in key or key in segment_path:
                    return data[key]
            
            return None
            
        except Exception as e:
            print(f"  Error retrieving text for {segment_path}: {e}")
            return None
    
    def build_dataset(self, query=None, start_date=None, end_date=None, 
                     state=None, max_results=100, search_type="PHRASE",
                     custom_url=None, include_snippet=True, snippet_query=None,
                     include_full_text=True):
        """
        Build a complete dataset with article metadata and full text.
        
        Args:
            query: Search term(s)
            start_date: Start date in YYYY-MM-DD format
            end_date: End date in YYYY-MM-DD format
            state: State to filter by
            max_results: Maximum number of results to retrieve
            search_type: Type of search ("PHRASE", "AND", "OR", "~5", "~10")
            custom_url: Use a pre-defined search URL from the Chronicling America website
            include_snippet: If True, get relevant snippet with highlighted terms
            snippet_query: Search terms to highlight in snippets (defaults to query)
            include_full_text: If True, also get full text (in addition to snippet)
            
        Returns:
            List of complete article records
        """
        # Search for articles
        articles = self.search_articles(query, start_date, end_date, state, 
                                       max_results, search_type, custom_url)
        
        # Determine snippet query
        if include_snippet and not snippet_query:
            snippet_query = query
        
        dataset = []
        
        print("\nRetrieving text for articles...")
        if include_full_text and include_snippet:
            print("(Getting both full text AND snippets - this will take longer)")
        
        for i, article in enumerate(articles, 1):
            print(f"\nProcessing article {i}/{len(articles)}")
            
            # Extract basic metadata
            record = {
                'id': article.get('id', ''),
                'title': article.get('title', ''),
                'date': article.get('date', ''),
                'url': article.get('url', ''),
                'newspaper_title': '',
                'location': '',
                'lccn': '',
                'page_number': '',
                'full_text': '',
                'relevant_snippet': '',
                'search_terms': '',
                'ocr_quality': ''
            }
            
            # Extract additional metadata
            if 'partof' in article:
                partof = article['partof']
                if isinstance(partof, list) and len(partof) > 0:
                    # partof can contain strings or dictionaries
                    if isinstance(partof[0], dict):
                        record['newspaper_title'] = partof[0].get('title', '')
                    elif isinstance(partof[0], str):
                        record['newspaper_title'] = partof[0]
            
            # Extract location from URL or metadata
            location_data = article.get('location', [])
            if location_data:
                record['location'] = ', '.join(location_data)
            
            # Try to get segment info and text
            segment_path, format_type = self.extract_segment_info(article)
            
            if segment_path == 'EMBEDDED':
                # Text is already in the article data
                record['full_text'] = article.get('full_text', '')
                record['ocr_quality'] = 'embedded'
                print(f"  Using embedded text ({len(record['full_text'])} characters)")
            elif segment_path:
                print(f"  Found segment: {segment_path}")
                
                # Get snippet if requested
                if include_snippet:
                    snippet_data = self.get_full_text(segment_path, format_type, 
                                                     snippet_query, include_snippet=True)
                    if snippet_data:
                        record['relevant_snippet'] = snippet_data.get('relevant_snippet', '')
                        search_terms = snippet_data.get('searchTerms', {})
                        record['search_terms'] = json.dumps(search_terms)
                        print(f"  Retrieved snippet ({len(record['relevant_snippet'])} characters)")
                
                # Get full text if requested
                if include_full_text:
                    full_text_data = self.get_full_text(segment_path, format_type, 
                                                        snippet_query, include_snippet=False)
                    if full_text_data:
                        record['full_text'] = full_text_data.get('full_text', '')
                        print(f"  Retrieved full text ({len(record['full_text'])} characters)")
                
                # Set OCR quality status
                if record['full_text'] or record['relevant_snippet']:
                    if record['full_text'] and record['relevant_snippet']:
                        record['ocr_quality'] = 'full_and_snippet'
                    elif record['full_text']:
                        record['ocr_quality'] = 'full_text_only'
                    else:
                        record['ocr_quality'] = 'snippet_only'
                else:
                    record['ocr_quality'] = 'failed'
                    print("  Failed to retrieve text")
            else:
                record['ocr_quality'] = 'no_segment'
                print("  No segment path found")
            
            dataset.append(record)
        
        return dataset
    
    def save_to_json(self, dataset, filename="dataset.json"):
        """Save dataset to JSON file."""
        filepath = self.output_dir / filename
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(dataset, f, indent=2, ensure_ascii=False)
        print(f"\nSaved {len(dataset)} records to {filepath}")
    
    def save_to_csv(self, dataset, filename="dataset.csv"):
        """Save dataset to CSV file."""
        filepath = self.output_dir / filename
        
        if not dataset:
            print("No data to save!")
            return
        
        # Get all unique keys
        keys = set()
        for record in dataset:
            keys.update(record.keys())
        keys = sorted(keys)
        
        with open(filepath, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=keys)
            writer.writeheader()
            writer.writerows(dataset)
        
        print(f"Saved {len(dataset)} records to {filepath}")
    
    def generate_statistics(self, dataset):
        """
        Generate statistics about the dataset.
        
        Args:
            dataset: List of article records
            
        Returns:
            Dictionary with statistics
        """
        if not dataset:
            print("No data to analyze!")
            return {}
        
        stats = {}
        
        # Total articles
        stats['total_articles'] = len(dataset)
        
        # OCR quality distribution
        ocr_quality = Counter(record.get('ocr_quality', 'unknown') for record in dataset)
        stats['ocr_quality'] = dict(ocr_quality)
        
        # Date distribution
        dates = [record.get('date', '') for record in dataset if record.get('date')]
        stats['date_range'] = {
            'earliest': min(dates) if dates else None,
            'latest': max(dates) if dates else None,
            'total_dates': len(set(dates))
        }
        
        # Year distribution
        years = []
        for record in dataset:
            date_str = record.get('date', '')
            if date_str:
                try:
                    year = date_str[:4]
                    years.append(year)
                except:
                    pass
        year_counts = Counter(years)
        stats['articles_per_year'] = dict(sorted(year_counts.items()))
        
        # Newspaper title distribution
        newspapers = [record.get('newspaper_title', 'Unknown') for record in dataset]
        newspaper_counts = Counter(newspapers)
        stats['articles_per_newspaper'] = dict(sorted(
            newspaper_counts.items(), 
            key=lambda x: x[1], 
            reverse=True
        ))
        
        # Location distribution
        locations = []
        for record in dataset:
            loc = record.get('location', '')
            if loc:
                locations.append(loc)
        location_counts = Counter(locations)
        stats['articles_per_location'] = dict(sorted(
            location_counts.items(), 
            key=lambda x: x[1], 
            reverse=True
        )[:20])  # Top 20 locations
        
        # Text statistics
        if any(r.get('full_text') for r in dataset):
            text_lengths = [len(record.get('full_text', '')) for record in dataset 
                          if record.get('full_text')]
            if text_lengths:
                stats['text_stats'] = {
                    'articles_with_text': len(text_lengths),
                    'avg_text_length': sum(text_lengths) / len(text_lengths),
                    'min_text_length': min(text_lengths),
                    'max_text_length': max(text_lengths)
                }
        
        if any(r.get('relevant_snippet') for r in dataset):
            snippet_lengths = [len(record.get('relevant_snippet', '')) for record in dataset 
                             if record.get('relevant_snippet')]
            if snippet_lengths:
                stats['snippet_stats'] = {
                    'articles_with_snippets': len(snippet_lengths),
                    'avg_snippet_length': sum(snippet_lengths) / len(snippet_lengths)
                }
        
        return stats
    
    def visualize_statistics(self, dataset, output_dir=None):
        """
        Create visualizations of the dataset statistics.
        
        Args:
            dataset: List of article records
            output_dir: Directory to save plots (defaults to self.output_dir)
        """
        if not dataset:
            print("No data to visualize!")
            return
        
        if output_dir is None:
            output_dir = self.output_dir
        else:
            output_dir = Path(output_dir)
            output_dir.mkdir(exist_ok=True)
        
        # Convert to DataFrame for easier manipulation
        df = pd.DataFrame(dataset)
        
        # Parse dates
        df['year'] = df['date'].apply(lambda x: x[:4] if x and len(x) >= 4 else None)
        df['year'] = pd.to_numeric(df['year'], errors='coerce')
        
        # Create a figure with multiple subplots
        fig = plt.figure(figsize=(16, 12))
        
        # 1. Articles over time (timeline)
        ax1 = plt.subplot(2, 2, 1)
        year_counts = df['year'].value_counts().sort_index()
        ax1.plot(year_counts.index, year_counts.values, marker='o', linewidth=2, markersize=6)
        ax1.set_xlabel('Year', fontsize=12)
        ax1.set_ylabel('Number of Articles', fontsize=12)
        ax1.set_title('Articles Distribution Over Time', fontsize=14, fontweight='bold')
        ax1.grid(True, alpha=0.3)
        
        # 2. Top newspapers (horizontal bar chart)
        ax2 = plt.subplot(2, 2, 2)
        newspaper_counts = df['newspaper_title'].value_counts().head(15)
        # Truncate long names
        labels = [name[:40] + '...' if len(name) > 40 else name 
                 for name in newspaper_counts.index]
        ax2.barh(range(len(newspaper_counts)), newspaper_counts.values)
        ax2.set_yticks(range(len(newspaper_counts)))
        ax2.set_yticklabels(labels, fontsize=9)
        ax2.set_xlabel('Number of Articles', fontsize=12)
        ax2.set_title('Top 15 Newspapers by Article Count', fontsize=14, fontweight='bold')
        ax2.invert_yaxis()
        
        # 3. Articles by decade
        ax3 = plt.subplot(2, 2, 3)
        df['decade'] = (df['year'] // 10 * 10).astype('Int64')
        decade_counts = df['decade'].value_counts().sort_index()
        colors = plt.cm.viridis(range(len(decade_counts)))
        ax3.bar(decade_counts.index.astype(str) + 's', decade_counts.values, color=colors)
        ax3.set_xlabel('Decade', fontsize=12)
        ax3.set_ylabel('Number of Articles', fontsize=12)
        ax3.set_title('Articles Distribution by Decade', fontsize=14, fontweight='bold')
        ax3.tick_params(axis='x', rotation=45)
        
        # 4. OCR Quality Distribution
        ax4 = plt.subplot(2, 2, 4)
        ocr_counts = df['ocr_quality'].value_counts()
        colors_pie = plt.cm.Set3(range(len(ocr_counts)))
        ax4.pie(ocr_counts.values, labels=ocr_counts.index, autopct='%1.1f%%',
                colors=colors_pie, startangle=90)
        ax4.set_title('OCR Quality Distribution', fontsize=14, fontweight='bold')
        
        plt.tight_layout()
        plt.savefig(output_dir / 'statistics_overview.png', dpi=300, bbox_inches='tight')
        print(f"\nSaved visualization to {output_dir / 'statistics_overview.png'}")
        plt.close()
        
        # Additional visualization: Detailed timeline with trend
        fig2, ax = plt.subplots(figsize=(14, 6))
        year_counts = df['year'].value_counts().sort_index()
        
        # Plot bars
        ax.bar(year_counts.index, year_counts.values, alpha=0.6, color='steelblue', 
               label='Articles per year')
        
        # Add trend line
        if len(year_counts) > 3:
            z = np.polyfit(year_counts.index, year_counts.values, 2)
            p = np.poly1d(z)
            ax.plot(year_counts.index, p(year_counts.index), "r--", 
                   linewidth=2, label='Trend', alpha=0.8)
        
        ax.set_xlabel('Year', fontsize=12)
        ax.set_ylabel('Number of Articles', fontsize=12)
        ax.set_title('Detailed Timeline: "Chinese students in American Newspapers', 
                    fontsize=14, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(output_dir / 'timeline_detailed.png', dpi=300, bbox_inches='tight')
        print(f"Saved detailed timeline to {output_dir / 'timeline_detailed.png'}")
        plt.close()
        
        # Location distribution map (if we have location data)
        if 'location_state' in df.columns and not df['location_state'].isna().all():
            fig3, ax = plt.subplots(figsize=(12, 8))
            
            # Extract states (they might be in lists)
            states = []
            for loc in df['location_state']:
                if isinstance(loc, list):
                    states.extend(loc)
                elif loc:
                    states.append(loc)
            
            state_counts = pd.Series(states).value_counts().head(20)
            
            ax.barh(range(len(state_counts)), state_counts.values, color='coral')
            ax.set_yticks(range(len(state_counts)))
            ax.set_yticklabels(state_counts.index)
            ax.set_xlabel('Number of Articles', fontsize=12)
            ax.set_title('Articles by State (Top 20)', fontsize=14, fontweight='bold')
            ax.invert_yaxis()
            
            plt.tight_layout()
            plt.savefig(output_dir / 'articles_by_state.png', dpi=300, bbox_inches='tight')
            print(f"Saved state distribution to {output_dir / 'articles_by_state.png'}")
            plt.close()
    
    def save_statistics_report(self, dataset, filename="statistics_report.txt"):
        """
        Save a text report of statistics.
        
        Args:
            dataset: List of article records
            filename: Output filename
        """
        stats = self.generate_statistics(dataset)
        filepath = self.output_dir / filename
        
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write("="*80 + "\n")
            f.write("CHRONICLING AMERICA DATASET STATISTICS REPORT\n")
            f.write("="*80 + "\n\n")
            
            f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
            
            f.write(f"Total Articles: {stats.get('total_articles', 0)}\n\n")
            
            # Date range
            if 'date_range' in stats:
                f.write("DATE RANGE:\n")
                f.write(f"  Earliest: {stats['date_range']['earliest']}\n")
                f.write(f"  Latest: {stats['date_range']['latest']}\n")
                f.write(f"  Unique dates: {stats['date_range']['total_dates']}\n\n")
            
            # OCR Quality
            if 'ocr_quality' in stats:
                f.write("OCR QUALITY DISTRIBUTION:\n")
                for quality, count in stats['ocr_quality'].items():
                    pct = (count / stats['total_articles']) * 100
                    f.write(f"  {quality}: {count} ({pct:.1f}%)\n")
                f.write("\n")
            
            # Articles per year (top 10)
            if 'articles_per_year' in stats:
                f.write("TOP 10 YEARS BY ARTICLE COUNT:\n")
                sorted_years = sorted(stats['articles_per_year'].items(), 
                                    key=lambda x: x[1], reverse=True)[:10]
                for year, count in sorted_years:
                    f.write(f"  {year}: {count} articles\n")
                f.write("\n")
            
            # Top newspapers
            if 'articles_per_newspaper' in stats:
                f.write("TOP 10 NEWSPAPERS BY ARTICLE COUNT:\n")
                for i, (newspaper, count) in enumerate(
                    list(stats['articles_per_newspaper'].items())[:10], 1):
                    f.write(f"  {i}. {newspaper}: {count} articles\n")
                f.write("\n")
            
            # Top locations
            if 'articles_per_location' in stats:
                f.write("TOP 10 LOCATIONS BY ARTICLE COUNT:\n")
                for i, (location, count) in enumerate(
                    list(stats['articles_per_location'].items())[:10], 1):
                    f.write(f"  {i}. {location}: {count} articles\n")
                f.write("\n")
            
            # Text statistics
            if 'text_stats' in stats:
                f.write("FULL TEXT STATISTICS:\n")
                for key, value in stats['text_stats'].items():
                    if isinstance(value, float):
                        f.write(f"  {key}: {value:.0f}\n")
                    else:
                        f.write(f"  {key}: {value}\n")
                f.write("\n")
            
            if 'snippet_stats' in stats:
                f.write("SNIPPET STATISTICS:\n")
                for key, value in stats['snippet_stats'].items():
                    if isinstance(value, float):
                        f.write(f"  {key}: {value:.0f}\n")
                    else:
                        f.write(f"  {key}: {value}\n")
                f.write("\n")
        
        print(f"\nSaved statistics report to {filepath}")

In [ ]:
# Example usage
if __name__ == "__main__":
    # Initialize scraper
    scraper = ChroniclingAmericaScraper(output_dir="chinese_students_data")
    
    # RECOMMENDED: Get BOTH full text AND snippets for complete analysis
    dataset = scraper.build_dataset(
        custom_url="https://www.loc.gov/collections/chronicling-america/?dl=page&ops=AND&qs=%22chinese+student%22&searchType=advanced",
        max_results=150,
        include_snippet=True,       # Get snippets with highlighted terms
        snippet_query="chinese student",
        include_full_text=True      # Also get full text (2x API calls per article)
    )
    
    # Alternative options:
    
    # Option 1: Only snippets (fastest)
    # dataset = scraper.build_dataset(
    #     custom_url="...",
    #     max_results=150,
    #     include_snippet=True,
    #     snippet_query="chinese student",
    #     include_full_text=False
    # )
    
    # Option 2: Only full text (no snippets)
    # dataset = scraper.build_dataset(
    #     custom_url="...",
    #     max_results=150,
    #     include_snippet=False,
    #     include_full_text=True
    # )
    
    # Save results
    scraper.save_to_json(dataset, "chinese_students_articles.json")
    scraper.save_to_csv(dataset, "chinese_students_articles.csv")
    
    # Generate statistics and visualizations
    print("\n" + "="*80)
    print("GENERATING STATISTICS AND VISUALIZATIONS")
    print("="*80)
    
    stats = scraper.generate_statistics(dataset)
    print("\nStatistics Summary:")
    print(f"  Total articles: {stats.get('total_articles', 0)}")
    print(f"  Date range: {stats.get('date_range', {}).get('earliest')} to {stats.get('date_range', {}).get('latest')}")
    print(f"  Unique newspapers: {len(stats.get('articles_per_newspaper', {}))}")
    
    # Save detailed statistics report
    scraper.save_statistics_report(dataset)
    
    # Create visualizations
    try:
        import numpy as np  # Required for trend line
        scraper.visualize_statistics(dataset)
    except ImportError as e:
        print(f"\nNote: Could not create visualizations. Missing library: {e}")
        print("Install with: pip install matplotlib pandas numpy")
    
    print("\nDataset building complete!")
    print(f"Articles with full text: {sum(1 for r in dataset if r.get('full_text'))}")
    print(f"Articles with snippets: {sum(1 for r in dataset if r.get('relevant_snippet'))}")
    print(f"Articles with both: {sum(1 for r in dataset if r.get('full_text') and r.get('relevant_snippet'))}")